# 🚇 NYC Transit Real-Time Passenger Flow & Overcrowding Analytics
## Exploratory Data Analysis, Entity Resolution, K-Means Clustering & Wait Assessment Correlation

**Học phần:** IT4931 - Lưu trữ và Xử lý Dữ liệu lớn (SoICT - HUST)  
Notebook này thực hiện khảo sát thực nghiệm trên tập dữ liệu thật **27 tuần (5.254.533 dòng)** trong `data/archive/`, kiểm chứng 5 thách thức kỹ thuật cốt lõi và hiển thị bộ biểu đồ từ **Batch Layer** & **Speed Layer**.
> **Lưu ý:** Notebook được thiết kế theo cơ chế **Zero-Heavy-Dependency** — chạy trực tiếp bằng Python có sẵn trên máy mà **không bắt buộc phải tạo `venv` hay tải các thư viện nặng**.

In [ ]:
import os
import sys
import json
from IPython.display import Image as IPyImage, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'src')):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.batch.entity_resolution import StationEntityResolver
print('Project Root:', PROJECT_ROOT)

## 1. Kiểm chứng Entity Resolution & Spatial Join (100% Match Rate)
Kiểm tra bảng ánh xạ `StationEntityResolver` giữa tên trạm viết tắt trong `TurnstileData` (`GRD CNTRL-42 ST`, `34 ST-PENN STA`, `42 ST-PORT AUTH`, `JFK JAMAICA CT1`) và danh mục không gian chuẩn `MTA_StationLocations.csv`.

In [ ]:
resolver = StationEntityResolver(os.path.join(PROJECT_ROOT, 'data/archive/MTA_StationLocations.csv'))
sample_stations = [
    ('GRD CNTRL-42 ST', '4567S', 'IRT'),
    ('34 ST-PENN STA', '123ACE', 'IRT'),
    ('42 ST-PORT AUTH', 'ACENQRS1237', 'IND'),
    ('47-50 STS ROCK', 'BDFM', 'IND'),
    ('ATL AV-BARCLAY', 'BDNQR2345', 'BMT'),
    ('JFK JAMAICA CT1', 'E', 'IND'),
    ('EXCHANGE PLACE', '1', 'PTH'),  # Non-MTA PATH station -> Filtered out
]
for st, ln, div in sample_stations:
    res = resolver.resolve(st, ln, div)
    if res:
        print(f"{st:<18} ({div}) -> ID={res['station_id']:<4} | {res['stop_name']:<38} | Borough={res['borough']} | ({res['latitude']}, {res['longitude']})")
    else:
        print(f"{st:<18} ({div}) -> FILTERED OUT (Non-MTA System)")

## 2. Kết quả Làm sạch Bộ đếm Lũy kế Full-Scale (27 Tuần - 5,25 Triệu Dòng)

In [ ]:
with open(os.path.join(PROJECT_ROOT, 'data/processed/clean_turnstile_summary.json'), encoding='utf-8') as f:
    clean_summary = json.load(f)
print(json.dumps(clean_summary, indent=2))
display(IPyImage(filename=os.path.join(PROJECT_ROOT, 'docs/figures/fig4_speed_layer_zscore_alerts.png')))

## 3. Phân cụm Hành vi Nhà ga bằng K-Means (437 Nhà ga | Silhouette = 0.6777)

In [ ]:
display(IPyImage(filename=os.path.join(PROJECT_ROOT, 'docs/figures/fig1_kmeans_24h_profiles.png')))
display(IPyImage(filename=os.path.join(PROJECT_ROOT, 'docs/figures/fig2_nyc_subway_geo_clusters.png')))

## 4. Phân tích Tương quan Quá tải Nhà ga với Chỉ số Trễ tàu (`Subway Wait Assessment` 2016)

In [ ]:
with open(os.path.join(PROJECT_ROOT, 'data/processed/correlation_results.json'), encoding='utf-8') as f:
    corr_res = json.load(f)
print(json.dumps(corr_res['summary_metrics'], indent=2))
display(IPyImage(filename=os.path.join(PROJECT_ROOT, 'docs/figures/fig3_wait_assessment_correlation.png')))